<a href="https://colab.research.google.com/github/shashidhar078/FlipGrid/blob/main/Model_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

train = pd.read_csv("/content/drive/MyDrive/train.csv")

# Drop Index
train = train.drop("Index", axis=1)

# Clean timestamp like 0:0, 13:30
train["timestamp"] = train["timestamp"].astype(str)

train["hour"] = train["timestamp"].str.split(":").str[0].astype(int)
train["minute"] = train["timestamp"].str.split(":").str[1].astype(int)

train = train.drop("timestamp", axis=1)

# Encode categorical columns
cat_cols = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather"]

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))

# X and y
X = train.drop("demand", axis=1)
y = train["demand"]

# Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train);

# Predict
pred = model.predict(X_valid)

# Scores
r2 = r2_score(y_valid, pred)
rmse = np.sqrt(mean_squared_error(y_valid, pred))
mae = mean_absolute_error(y_valid, pred)

print("R2 Score:", r2)
print("Competition Score:", max(0, 100*r2))
print("RMSE:", rmse)
print("MAE:", mae)

R2 Score: 0.9409314067825232
Competition Score: 94.09314067825233
RMSE: 0.03457198595004924
MAE: 0.022650445414869737


In [7]:
print(train["demand"].describe())

count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [8]:
print(train.columns)

Index(['geohash', 'day', 'demand', 'RoadType', 'NumberofLanes',
       'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour',
       'minute'],
      dtype='object')


In [9]:
print("demand" in X.columns)

False


In [10]:
train_pred = model.predict(X_train)

print("Train R2 :", r2_score(y_train, train_pred))
print("Valid R2 :", r2_score(y_valid, pred))

Train R2 : 0.9917799663792387
Valid R2 : 0.9409314067825232


In [15]:
!pip install catboost
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

train = pd.read_csv("/content/drive/MyDrive/train.csv")

y = train["demand"]
X = train.drop(["demand", "timestamp", "Index"], axis=1) # Drop 'Index' as it's not a feature

# Fill NaN values in categorical features with 'Unknown'
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("Unknown").astype(str)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, val_idx in kf.split(X):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function="RMSE",
        verbose=0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    pred = model.predict(X_val)

    score = r2_score(y_val, pred)
    scores.append(score)

print("Fold Scores:", scores)
print("CV SCORE:", sum(scores) / len(scores))

Fold Scores: [0.8799982470477905, 0.878720539061351, 0.8854721463182126, 0.8718471719206052, 0.8804931149269264]
CV SCORE: 0.8793062438549771


In [3]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# LOAD DATA
# ==========================================

train = pd.read_csv("/content/drive/MyDrive/train.csv")

# ==========================================
# DROP TIMESTAMP
# ==========================================

train = train.drop("timestamp", axis=1)

# ==========================================
# ENCODE CATEGORICAL FEATURES
# ==========================================

cat_cols = train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))

# ==========================================
# TARGET
# ==========================================

y = train["demand"]
X = train.drop("demand", axis=1)

# ==========================================
# K FOLD
# ==========================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, val_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    score = r2_score(y_val, pred)

    scores.append(score)

print("Fold Scores:", scores)
print("CV SCORE:", sum(scores) / len(scores))
print("Competition Score:", (sum(scores) / len(scores)) * 100)

Fold Scores: [0.9440620886582206, 0.9440337072743574, 0.9460928449729505, 0.93794116837465, 0.9476825288591836]
CV SCORE: 0.9439624676278724
Competition Score: 94.39624676278724
